## နှစ်ယောက် တန်းစီထားသော အေးဂျင့်များ:

1. **ရှေ့ကွင်းဧည့်ခံ အေးဂျင့်**: မြို့အတွက် အစပိုင်းဆွဲဆောင်မှု အကြံပြုချက်များ ပေးသည်  
2. **အောက်ခံ အေးဂျင့်**: ရှေ့ကွင်းဧည့်ခံ အကြံပြုချက်ကို လူကြိုက်ဆုံးမှတ်ချက်အရ ပြန်လည်သုံးသပ် အသုံးချသည်  

## တန်းစီသွားသော ပုံစံအခြေခံ အကျိုးကျေးဇူးများ:

- **အဆင့်ဆင့်တိုးတက်မှု**: ဒုတိယအေးဂျင့်သည် ပထမအေးဂျင့်၏ လုပ်ငန်းထက် တိုးတက်အောင်လုပ်ဆောင်သည်  
- **အထူးပြုခြင်း**: အေးဂျင့်တိုင်းမှာ သတ်မှတ်ထားသော လုပ်ငန်းတာဝန်တစ်ရပ်ခုခံထားသည်  
- **အရည်အသွေး ထိန်းချုပ်မှု**: အတွင်းရှိ သုံးသပ်ခြင်းနှင့် စစ်ဆေးခြင်း ခြေလှမ်းပါရှိသည်  
- **သေချာသော သတင်းအချက်အလက် စီးဆင်းမှု**: အေးဂျင့်များအကြား ဖွဲ့စည်းထားသော လက်လှမ်းလွှဲပေးမှု  

## ကြိုတင်လိုအပ်ချက်များ:  
- Microsoft Agent Framework ထည့်သွင်းပြီးသား  
- Azure AI Foundry project endpoint နှင့် မော်ဒယ် တပ်ဆင်မှုစနစ် ပြင်ဆင်ပြီး (`AZURE_AI_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`)  
- Azure CLI ဖြင့် အသိအမှတ်ပြုထားပြီး (`az login`)  
- အေးဂျင့် အခြေခံအယူအဆများ နားလည်မှုရှိပါရန်


In [ ]:
import asyncio
import json
import os
from typing import Any, cast

from agent_framework import Message, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("All imports successful!")

## Step 1: Define Pydantic Models for Structured Outputs

ဤမော်ဒယ်များသည် agent တစ်ဦးချင်းစီမှ ပြန်ပေးပို့မည့် schema ကို သတ်မှတ်ပေးသည်။ front desk agent သည် အကြံပြုချက်တစ်ခု ပေးပြီး၊ concierge agent သည် ပြန်လည်သုံးသပ်ချက်နှင့် အဆင့်သတ်မှတ်ချက် ပေးသည်။


In [ ]:
class AttractionRecommendation(BaseModel):
    """Attraction recommendation from the front desk agent."""

    city: str
    attraction_name: str
    description: str
    category: str  # e.g., "museum", "landmark", "park", "entertainment"
    recommended_duration: str  # e.g., "2-3 hours", "half day"
    why_recommended: str
    best_time_to_visit: str


class AttractionReview(BaseModel):
    """Expert review and rating from the concierge agent."""

    attraction_name: str
    city: str
    popularity_score: int  # 1-10 scale
    popularity_reasoning: str
    visitor_rating: float  # 1.0-5.0 scale
    pros: list[str]
    cons: list[str]
    concierge_recommendation: str
    alternative_suggestions: list[str]

## အဆင့် ၂: ပတ်ဝန်းကျင်အပြောင်းအလဲများကို Load လုပ်ပြီး Foundry Provider ကို ဖွဲ့စည်းပါ

သင်ခန်းစာ ၀၁–၁၃ တွင် အသုံးပြုသည့် ပုံစံနှင့် ကိုက်ညီစေရန် keyless `AzureCliCredential` အတည်ပြုမှုဖြင့် `AzureAIProjectAgentProvider` ကို အသုံးပြုပါ။


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("Azure AI Foundry provider configured successfully!")

## အဆင့် ၃: ဆက်တိုက်လည်ပတ်သော အေးဂျင့် နှစ်ဦး ဖန်တီးခြင်း

အေးဂျင့်တစ်ဦးစီသည် ဆက်တိုက်လည်ပတ်မှုလုပ်ငန်းစဉ်တွင် သတ်မှတ်ထားသည့် အခန်းကဏ္ဍတစ်ခုရှိသည်။ မျက်နှာစာအေးဂျင့်သည် အကြံပြုချက်များပေးပြီး၊ ကွန်ဆီယား့ အေးဂျင့်သည် ထိုအကြံပြုချက်များအား ပြန်လည်သုံးသပ်ပြီး အဆင့်သတ်မှတ်ပေးသည်။


In [ ]:
# Agent 1: Front Desk Agent (Makes initial recommendations)
front_desk_agent = await provider.create_agent(
    name="front-desk-agent",
    instructions=(
        "You are a knowledgeable hotel front desk agent who specializes in local attractions. "
        "When a guest asks about attractions in a city, provide a single, well-researched recommendation "
        "for a popular tourist attraction. Focus on giving practical information including what makes "
        "this attraction special, how long to spend there, and the best time to visit. "
        "Be helpful and enthusiastic about your recommendation. "
        "Return structured JSON matching the AttractionRecommendation schema."
    ),
)

# Agent 2: Concierge Agent (Reviews and rates recommendations)
concierge_agent = await provider.create_agent(
    name="concierge-agent",
    instructions=(
        "You are an expert concierge with extensive knowledge of tourist attractions worldwide. "
        "You will receive an attraction recommendation and must provide an expert review and rating. "
        "Evaluate the recommendation based on the attraction's popularity, visitor satisfaction, "
        "and overall quality. Provide a popularity score (1-10), visitor rating (1.0-5.0), "
        "list pros and cons, and give your professional assessment. "
        "Also suggest alternative attractions if appropriate. "
        "Return structured JSON matching the AttractionReview schema."
    ),
)


## အဆင့် ၄: အဆက်မပြတ် လုပ်ငန်းစဉ် တည်ဆောက်ခြင်း

`WorkflowBuilder` သည် အောက်ပါအတိုင်း လုပ်ငန်းစဉ်တစ်ခုကို ဖန်တီးသည်-
1. **Front Desk Agent** သည် အသုံးပြုသူ၏ အချက်အလက်ကို လက်ခံပြီး အကြံပြုချက် တစ်ရပ် ပေးပို့သည်
2. **Concierge Agent** သည် front desk အကြံပြုချက်ကို လက်ခံပြီး ပညာရှင်သုံးသပ်ချက် ပေးသည်
3. **Output** တွင် မူလအကြံပြုချက် နှင့် ပညာရှင်သုံးသပ်ချက် နှစ်ခုလုံး ပါဝင်သည်


In [ ]:
# Build the sequential workflow with WorkflowBuilder
workflow = (
    WorkflowBuilder(
        start_executor=front_desk_agent,
        output_executors=[front_desk_agent, concierge_agent],
    )
    .add_edge(front_desk_agent, concierge_agent)
    .build()
)

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Sequential Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Flow:</strong><br>
        • User Input → <strong>Front Desk Agent</strong> (recommendation)<br>
        • Front Desk Output → <strong>Concierge Agent</strong> (review & rating)<br>
        • Final Output → Combined recommendation + expert review
    </p>
</div>
"""))

In [ ]:
async def display_attraction_recommendation(city: str):
    """Run the sequential workflow and display formatted results."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Processing Attraction Recommendation for {city}</h3>
        <p style='margin: 0;'><strong>Status:</strong> Running sequential workflow...</p>
    </div>
    """))

    # Run the workflow. With WorkflowBuilder(output_executors=[a1, a2]),
    # outputs is a list of AgentResponse objects, one per output executor.
    events = await workflow.run(f"I want to visit an attraction in {city}")
    outputs = events.get_outputs()

    front_desk_response = outputs[0].text if len(outputs) > 0 else None
    concierge_response = outputs[1].text if len(outputs) > 1 else None

    # Display results
    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Attraction Recommendation for {city}</h2>
        <p style='margin: 0; font-size: 14px; opacity: 0.9;'>Generated by sequential agent workflow</p>
    </div>
    """))

    # Process and display responses
    if front_desk_response:
        try:
            recommendation_data = AttractionRecommendation.model_validate_json(front_desk_response)
            display_front_desk_section(recommendation_data)
        except Exception as e:
            display(HTML(f"""
            <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>Error parsing front desk response:</strong> {str(e)}
                <details><summary>Raw response</summary>{front_desk_response}</details>
            </div>
             """))

    if concierge_response:
        try:
            review_data = AttractionReview.model_validate_json(concierge_response)
            display_concierge_section(review_data)
        except Exception as e:
            display(HTML(f"""
            <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>Error parsing concierge response:</strong> {str(e)}
                <details><summary>Raw response</summary>{concierge_response}</details>
            </div>
            """))


def display_front_desk_section(data: AttractionRecommendation):
    """Display front desk recommendation in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏨 Front Desk Recommendation</h3>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>{data.attraction_name}</h4>
            <span style='background: #2196f3; color: white; padding: 4px 8px; border-radius: 12px; font-size: 12px;'>{data.category}</span>
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Description:</strong> {data.description}
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Why Recommended:</strong> {data.why_recommended}
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Recommended Duration:</strong> {data.recommended_duration}
        </div>
        <div>
            <strong style='color: #333;'>Best Time to Visit:</strong> {data.best_time_to_visit}
        </div>
    </div>
    """))


def display_concierge_section(data: AttractionReview):
    """Display concierge review in a formatted section."""

    # Create star rating display
    star_rating = "⭐" * int(data.visitor_rating) + "☆" * (5 - int(data.visitor_rating))

    # Create popularity bar
    popularity_bar = "🟩" * data.popularity_score + "⬜" * (10 - data.popularity_score)

    pros_list = "".join([f"<li style='color: #4caf50;'>✓ {pro}</li>" for pro in data.pros])
    cons_list = "".join([f"<li style='color: #f44336;'>✗ {con}</li>" for con in data.cons])
    alternatives_list = "".join([f"<li>{alt}</li>" for alt in data.alternative_suggestions])

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🎩 Concierge Expert Review</h3>

        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
            <div style='background: rgba(255,152,0,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 8px 0; color: #333;'>Popularity Score</h4>
                <div style='font-size: 24px; font-weight: bold; color: #f57c00;'>{data.popularity_score}/10</div>
                <div style='font-size: 12px; margin-top: 5px;'>{popularity_bar}</div>
            </div>
            <div style='background: rgba(255,152,0,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 8px 0; color: #333;'>Visitor Rating</h4>
                <div style='font-size: 20px; font-weight: bold; color: #f57c00;'>{data.visitor_rating}/5.0</div>
                <div style='font-size: 16px; margin-top: 5px;'>{star_rating}</div>
            </div>
        </div>

        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Popularity Reasoning:</strong> {data.popularity_reasoning}
        </div>

        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 15px;'>
            <div>
                <h4 style='margin: 0 0 8px 0; color: #333;'>Pros:</h4>
                <ul style='margin: 0; padding-left: 20px;'>{pros_list}</ul>
            </div>
            <div>
                <h4 style='margin: 0 0 8px 0; color: #333;'>Cons:</h4>
                <ul style='margin: 0; padding-left: 20px;'>{cons_list}</ul>
            </div>
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Concierge Recommendation:</strong> {data.concierge_recommendation}
        </div>

        <div>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Alternative Suggestions:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{alternatives_list}</ul>
        </div>
    </div>
    """))


# Test with Stockholm
await display_attraction_recommendation("Stockholm")


## အဆင့် ၈: လုပ်ငန်းစဉ်ခွဲခြမ်းစိတ်ဖြာခြင်း - အဆက်မပြတ်စီးဆင်းမှုကို နားလည်ခြင်း

ရာထူးဆိုင်ရာများအကြား သတင်းအချက်အလက် မည်သို့စီးဆင်းသည်ကို စိစစ်ကြည့်ပြီး ဆွေးနွေးမှုမှတ်တမ်းကို ခွဲခြမ်းစိတ်ဖြာကြမည်။


In [ ]:
async def analyze_sequential_flow(city: str):
    """Analyze the sequential flow between agents."""

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>Sequential Flow Analysis for {city}</h3>
        <p style='margin: 0;'>Examining agent interactions and information handoff...</p>
    </div>
    """))

    # Run the workflow
    user_input = f"I want to visit an attraction in {city}"
    events = await workflow.run(user_input)
    outputs = events.get_outputs()

    # Reconstruct the conversation flow as a list of (author, text) steps.
    # outputs is a list of AgentResponse objects, ordered to match output_executors.
    steps = [("user", user_input)]
    if len(outputs) > 0:
        steps.append(("front-desk-agent", outputs[0].text))
    if len(outputs) > 1:
        steps.append(("concierge-agent", outputs[1].text))

    display(HTML(f"""
    <div style='padding: 25px; background: #f3e5f5; border-radius: 12px; margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0; color: #7b1fa2;'>Conversation Flow Analysis</h2>
    </div>
    """))

    # Display each step in the sequence
    for i, (author, text) in enumerate(steps, 1):
        role_color = {
            "user": "#2196f3",
            "front-desk-agent": "#4caf50",
            "concierge-agent": "#ff9800"
        }.get(author, "#666666")

        role_name = {
            "user": "👤 User",
            "front-desk-agent": "🏨 Front Desk Agent",
            "concierge-agent": "🎩 Concierge Agent"
        }.get(author, "Unknown")

        # Truncate long messages for flow analysis
        content_preview = text[:200] + "..." if len(text) > 200 else text
        display(HTML(f"""
        <div style='padding: 15px; background: white; border-left: 4px solid {role_color}; border-radius: 4px; margin: 10px 0; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <div style='display: flex; align-items: center; margin-bottom: 10px;'>
                <span style='font-weight: bold; color: {role_color}; margin-right: 10px;'>Step {i}:</span>
                <span style='font-weight: bold; color: {role_color};'>{role_name}</span>
            </div>
            <div style='color: #555; font-size: 14px; line-height: 1.4;'>
                {content_preview}
            </div>
        </div>
        """))

    # Analyze the flow
    display(HTML(f"""
    <div style='padding: 20px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 15px 0;'>Flow Analysis Summary</h3>
        <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
            <li><strong>Total Steps:</strong> {len(steps)}</li>
            <li><strong>Agents Involved:</strong> 2 (Front Desk + Concierge)</li>
            <li><strong>Flow Pattern:</strong> Linear sequential (User → Agent 1 → Agent 2)</li>
            <li><strong>Information Handoff:</strong> Front desk recommendation becomes concierge input</li>
            <li><strong>Output Quality:</strong> Enhanced through expert review and rating</li>
        </ul>
    </div>
    """))


# Analyze the flow for Barcelona
await analyze_sequential_flow("Barcelona")


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
